# Serving — agg_channel_ranking (dashboard v3, added 2026-11-03)

Page 3 "Kênh review" of the dashboard. **Grain:** 1 YouTube channel with at least `MIN_REVIEW_VIDEOS` (3) restaurant review videos (`dim_video.video_type = 'restaurant_review'`). **Full overwrite** every run. Run after `gold/agg_daily_snapshots` (`agg_channel_daily`) and `gold/agg_restaurant_ranking` (Google rating of the restaurants).

- Latest `subscriber_count`, `view_count`, `video_count` from `agg_channel_daily`.
- `views_7d` / `subscribers_gained_7d` = latest count − the last daily count on or before 7 days earlier (no count before the window: since the first count); `views_prev_7d` = the 7 days before; `growth_pct_7d` when `views_prev_7d` ≥ 10,000. `new_videos_7d` = review videos published in the window. `trending_rank` = rank by `views_7d` (channels with `views_7d` > 0).
- `restaurants_reviewed` = distinct located restaurants its videos mention (`fact_video_restaurant_mention`); `rated_restaurants` / `avg_restaurant_rating` = those with a Google rating.
- **`credibility_score`** (0–100, datle 2026-11-03) = 100 × (0.35 × subscriber part + 0.25 × view part + 0.40 × quality part): subscriber part = log(1 + subs) ÷ log(1 + 10,000,000), view part = log(1 + views) ÷ log(1 + 1,000,000,000), both capped at 1; quality part = (average Google rating of the restaurants it reviewed − 3.5) ÷ 1.5, clipped 0–1, only with ≥ 3 rated restaurants (else a neutral 0.5, so a big channel with no evidence of good picks does not rank first). `credibility_rank` ranks by it.
- `restaurants_text` = up to 5 restaurants it mentions (best `restaurant_score` first); `comments_analyzed`, `positive_share`, `negative_share` = sentiment (`stg/comment_sentiment`, else the `stg/comment_reaction` label) of the comments on its review videos (datle 2026-11-03).


In [ ]:
for layer in ("bronze", "silver", "gold"):
    dbutils.widgets.text(layer, f"prod.{layer}")
BRONZE, SILVER, GOLD = (dbutils.widgets.get(layer) for layer in ("bronze", "silver", "gold"))
dbutils.widgets.text("serving", "prod.serving")
SERVING = dbutils.widgets.get("serving")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SERVING}")
TARGET = f"{SERVING}.agg_channel_ranking"
RANKING = f"{SERVING}.agg_restaurant_ranking"
MIN_REVIEW_VIDEOS = 3
TREND_DAYS = 7
GROWTH_MIN_PREV_VIEWS = 10000
CRED_WEIGHTS = {"subs": 0.35, "views": 0.25, "quality": 0.40}
SUBS_REF, VIEWS_REF = 10_000_000, 1_000_000_000   # 1M / 100M saturated: every big channel scored 100
MIN_RATED_RESTAURANTS = 3
TOP_RESTAURANTS = 5   # restaurants_text: best scored restaurants the channel mentions
NEUTRAL_QUALITY = 0.5   # fewer rated restaurants: neutral, so a big channel with no evidence is not ranked first
W = CRED_WEIGHTS
print(f"Serving target: {TARGET}")REACTION_SRC = f"{SILVER}.stg_comment_reaction"   # stg/comment_reaction: fills the sentiment of trending-video comments
if not spark.catalog.tableExists(REACTION_SRC):
    REACTION_SRC = "(SELECT CAST(NULL AS STRING) AS comment_id, CAST(NULL AS STRING) AS reaction WHERE false)"


In [ ]:
spark.sql(f"""
    CREATE OR REPLACE TABLE {TARGET}
    COMMENT 'Serving: 1 YouTube channel with >= {MIN_REVIEW_VIDEOS} restaurant review videos: size, 7-day trend, restaurants reviewed, credibility score. Full rebuild.'
    AS
    WITH d AS (SELECT channel_key, to_date(CAST(snapshot_date_key AS STRING), 'yyyyMMdd') AS day,
                      subscriber_count, view_count, video_count
               FROM {SERVING}.agg_channel_daily WHERE view_count IS NOT NULL),
    w AS (SELECT MAX(day) AS today, date_sub(MAX(day), {TREND_DAYS}) AS start, date_sub(MAX(day), {2 * TREND_DAYS}) AS prev_start FROM d),
    latest AS (SELECT channel_key, max_by(subscriber_count, day) AS subscriber_count, max_by(view_count, day) AS view_count,
                      max_by(video_count, day) AS video_count, min_by(view_count, day) AS first_views,
                      min_by(subscriber_count, day) AS first_subs
               FROM d GROUP BY channel_key),
    at_start AS (SELECT d.channel_key, max_by(d.view_count, d.day) AS views_start, max_by(d.subscriber_count, d.day) AS subs_start
                 FROM d CROSS JOIN w WHERE d.day <= w.start GROUP BY d.channel_key),
    at_prev AS (SELECT d.channel_key, max_by(d.view_count, d.day) AS views_prev_start
                FROM d CROSS JOIN w WHERE d.day <= w.prev_start GROUP BY d.channel_key),
    vids AS (   -- restaurant review videos per channel
        SELECT v.channel_id, COUNT(*) AS review_videos,
               COUNT_IF(to_date(v.published_at) > w.start) AS new_videos_7d
        FROM {GOLD}.dim_video v CROSS JOIN w
        WHERE v.video_type = 'restaurant_review'
        GROUP BY v.channel_id
    ),
    rest AS (   -- located restaurants its videos mention, with the Google rating
        SELECT v.channel_id, COUNT(DISTINCT m.restaurant_id) AS restaurants_reviewed,
               COUNT(DISTINCT CASE WHEN r.google_rating IS NOT NULL THEN m.restaurant_id END) AS rated_restaurants,
               AVG(r.google_rating) AS avg_restaurant_rating,
               -- the restaurants it mentions most (then best score), for the dashboard
               array_join(slice(transform(array_sort(collect_set(named_struct(
                   'neg_score', -coalesce(r.restaurant_score, 0), 'name', r.restaurant_name))), x -> x.name), 1, {TOP_RESTAURANTS}), ' · ') AS restaurants_text
        FROM {GOLD}.fact_video_restaurant_mention m
        JOIN {GOLD}.dim_video v ON v.video_key = m.video_key
        JOIN (SELECT DISTINCT restaurant_id, google_rating, restaurant_score, restaurant_name FROM {RANKING}) r ON r.restaurant_id = m.restaurant_id
        WHERE m.restaurant_id <> 'unknown'
        GROUP BY v.channel_id
    ),
    senti AS (   -- audience sentiment of the analyzed comments on its review videos
        SELECT v.channel_id, COUNT(*) AS comments_analyzed,
               COUNT_IF(se.sentiment = 'positive') / COUNT(*) AS positive_share,
               COUNT_IF(se.sentiment = 'negative') / COUNT(*) AS negative_share
        FROM {GOLD}.dim_comment c
        JOIN {GOLD}.dim_video v ON v.video_key = c.video_key AND v.video_type = 'restaurant_review'
        JOIN (SELECT coalesce(s.comment_id, r.comment_id) AS comment_id,
                     coalesce(s.sentiment, CASE r.reaction WHEN 'complaint' THEN 'negative'
                                                           WHEN 'ask_info' THEN 'neutral' WHEN 'other' THEN 'neutral'
                                                           ELSE 'positive' END) AS sentiment
              FROM {SILVER}.stg_comment_sentiment s
              FULL OUTER JOIN {REACTION_SRC} r ON r.comment_id = s.comment_id) se ON se.comment_id = c.comment_id
        WHERE c.comment_key <> -1
        GROUP BY v.channel_id
    ),
    base AS (
        SELECT ch.channel_id, l.channel_key, ch.channel_title, ch.thumbnail_url, ch.custom_url,
               concat('https://www.youtube.com/channel/', ch.channel_id) AS channel_url,
               l.subscriber_count, l.view_count, l.video_count,
               vi.review_videos, vi.new_videos_7d,
               coalesce(re.restaurants_reviewed, 0) AS restaurants_reviewed,
               coalesce(re.rated_restaurants, 0) AS rated_restaurants, re.avg_restaurant_rating,
               re.restaurants_text, coalesce(sn.comments_analyzed, 0) AS comments_analyzed,
               sn.positive_share, sn.negative_share,
               greatest(l.view_count - coalesce(s.views_start, l.first_views), 0) AS views_7d,
               CASE WHEN s.views_start IS NOT NULL AND p.views_prev_start IS NOT NULL
                    THEN greatest(s.views_start - p.views_prev_start, 0) END AS views_prev_7d,
               l.subscriber_count - coalesce(s.subs_start, l.first_subs) AS subscribers_gained_7d,
               least(ln(1 + coalesce(l.subscriber_count, 0)) / ln(1 + {SUBS_REF}), 1) AS subs_part,
               least(ln(1 + coalesce(l.view_count, 0)) / ln(1 + {VIEWS_REF}), 1) AS views_part,
               CASE WHEN coalesce(re.rated_restaurants, 0) >= {MIN_RATED_RESTAURANTS}
                    THEN least(greatest((re.avg_restaurant_rating - 3.5) / 1.5, 0), 1) END AS quality_part
        FROM latest l
        CROSS JOIN w
        JOIN {GOLD}.dim_channel ch ON ch.channel_key = l.channel_key
        JOIN vids vi ON vi.channel_id = ch.channel_id AND vi.review_videos >= {MIN_REVIEW_VIDEOS}
        LEFT JOIN rest re ON re.channel_id = ch.channel_id
        LEFT JOIN senti sn ON sn.channel_id = ch.channel_id
        LEFT JOIN at_start s ON s.channel_key = l.channel_key
        LEFT JOIN at_prev p ON p.channel_key = l.channel_key
    ),
    scored AS (
        SELECT *,
               round(100 * ({W['subs']} * subs_part + {W['views']} * views_part
                            + {W['quality']} * coalesce(quality_part, {NEUTRAL_QUALITY})), 1) AS credibility_score
        FROM base
    )
    SELECT channel_id, channel_key, channel_title, thumbnail_url, custom_url, channel_url,
           subscriber_count, view_count, video_count,
           CAST(review_videos AS INT) AS review_videos, CAST(new_videos_7d AS INT) AS new_videos_7d,
           CAST(restaurants_reviewed AS INT) AS restaurants_reviewed, CAST(rated_restaurants AS INT) AS rated_restaurants,
           round(avg_restaurant_rating, 2) AS avg_restaurant_rating,
           restaurants_text, CAST(comments_analyzed AS INT) AS comments_analyzed,
           round(positive_share, 4) AS positive_share, round(negative_share, 4) AS negative_share,
           CAST(views_7d AS BIGINT) AS views_7d, CAST(views_prev_7d AS BIGINT) AS views_prev_7d,
           CASE WHEN views_prev_7d >= {GROWTH_MIN_PREV_VIEWS} THEN round((views_7d - views_prev_7d) / views_prev_7d, 4) END AS growth_pct_7d,
           CAST(subscribers_gained_7d AS BIGINT) AS subscribers_gained_7d,
           CASE WHEN views_7d > 0 THEN CAST(RANK() OVER (PARTITION BY views_7d > 0 ORDER BY views_7d DESC, view_count DESC) AS INT) END AS trending_rank,
           round(subs_part, 3) AS subs_part, round(views_part, 3) AS views_part, round(quality_part, 3) AS quality_part,
           credibility_score,
           CAST(RANK() OVER (ORDER BY credibility_score DESC, subscriber_count DESC) AS INT) AS credibility_rank,
           current_timestamp() AS etl_date
    FROM scored
""")
s = spark.sql(f"SELECT COUNT(*) AS n, COUNT(DISTINCT channel_id) AS grain, COUNT(quality_part) AS with_quality FROM {TARGET}").collect()[0]
print(f"📊 {TARGET}: {s['n']} channels ({s['with_quality']} with the restaurant quality part) "
      + ("✅" if s["n"] == s["grain"] else "❌ duplicate channel_id"))
assert s["n"] == s["grain"], "duplicate channel_id in agg_channel_ranking"
display(spark.sql(f"""SELECT credibility_rank, channel_title, subscriber_count, view_count, review_videos, restaurants_reviewed,
                             avg_restaurant_rating, credibility_score FROM {TARGET} ORDER BY credibility_rank LIMIT 15"""))
display(spark.sql(f"""SELECT trending_rank, channel_title, views_7d, subscribers_gained_7d, new_videos_7d, growth_pct_7d
                      FROM {TARGET} WHERE trending_rank IS NOT NULL ORDER BY trending_rank LIMIT 15"""))